# Climate MVT: Cloud-Optimized GeoTIFF Generation

This notebook generates Cloud-Optimized GeoTIFF (COG) files for climate data visualization.

**What this does:**
1. Load climate grid data from PostgreSQL
2. Compute global historical min/max across all times
3. Generate progressive zoom levels (z0-z5) with increasing resolution
4. Apply configurable colormap to Band 1–3 (visual RGB) — default: `vanimo`
5. Normalize to grayscale for Band 2 (raw data)
6. Create alpha channel (Band 5) for transparency (null data)
7. Embed WGS84 georeferencing and metadata
8. Save COGs to `/local_only/climate_mvt/{variable}/{time}/z{0-5}.tif`

**Output:**
- 5-band Cloud-Optimized GeoTIFF files ready for DeckGL BitmapLayer consumption
- Band 1: Red (colormapped)
- Band 2: Green (colormapped)
- Band 3: Blue (colormapped)
- Band 4: Grayscale (raw data, 0-255 normalized)
- Band 5: Alpha (255=valid data, 0=transparent/null)
- Full GeoTIFF metadata tags embedded

## Setup & Configuration

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
from pyproj import Transformer
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Add project root to path so app.utils can be imported from the notebook
sys.path.insert(0, str(Path.cwd().parent))
from app.utils.colormap_utils import apply_colormap, DEFAULT_COLORMAP, list_colormaps

print("✓ Libraries imported successfully")
print(f"✓ Colormap utility loaded — default: {DEFAULT_COLORMAP}")

In [ ]:
# Database configuration
DB_PARAMS = {
    "host": "localhost",
    "user": "leon",
    "password": "leon",
    "database": "oc-database",
    "port": 5432,
}

# Output configuration
COG_BASE_DIR = Path.cwd().parent / "local_only" / "climate_mvt"
COG_BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Database: {DB_PARAMS['database']}")
print(f"✓ COG output directory: {COG_BASE_DIR}")

# Australia bounds (WGS84, from actual database query)
# These are the EXACT bounds of the climate data in the database
AUSTRALIA_BOUNDS_WGS84 = {
    "west": 112.900000,      # Actual min_lon
    "south": -43.650000,     # Actual min_lat
    "east": 153.650000,      # Actual max_lon (corrected from 154.0)
    "north": -10.050000,     # Actual max_lat (corrected from -10.0)
}

print(f"✓ Bounds (WGS84): {AUSTRALIA_BOUNDS_WGS84}")

# Create transformer for WGS84 -> Web Mercator reprojection
# This will be used to transform bounds to Web Mercator (EPSG:3857)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# Transform bounds to Web Mercator (coordinates will be in meters, not degrees)
minx_wm, miny_wm = transformer.transform(AUSTRALIA_BOUNDS_WGS84['west'], AUSTRALIA_BOUNDS_WGS84['south'])
maxx_wm, maxy_wm = transformer.transform(AUSTRALIA_BOUNDS_WGS84['east'], AUSTRALIA_BOUNDS_WGS84['north'])

AUSTRALIA_BOUNDS_WEB_MERCATOR = {
    "west": minx_wm,
    "south": miny_wm,
    "east": maxx_wm,
    "north": maxy_wm,
}

print(f"✓ Bounds (Web Mercator): {AUSTRALIA_BOUNDS_WEB_MERCATOR}")
print(f"  Note: Web Mercator coordinates are in meters, not degrees")

In [ ]:
# Zoom level specifications
# NOTE: Capped at z5 (8192x8192). The source climate grid (~0.05deg resolution)
# only has ~820x680 native data points over Australia. z5 already provides
# ~10x oversampling, so higher zoom levels add no information and consume
# excessive memory (z6 would need ~8-10 GB RAM).
ZOOM_SPECS = {
    0: (256, 256),
    1: (512, 512),
    2: (1024, 1024),
    3: (2048, 2048),
    4: (4096, 4096),
    5: (8192, 8192),
}

print(f"✓ Zoom levels configured: z0 (256×256) through z5 (8192×8192)")

# ── Colormap selection ─────────────────────────────────────────────────
# Set COLORMAP_NAME to any matplotlib colormap name.
# Run list_colormaps() in any cell to see curated options.
# Default is 'vanimo': dark-mode diverging (Crameri scientific colormaps).
COLORMAP_NAME = DEFAULT_COLORMAP  # e.g. 'vanimo', 'viridis', 'RdBu', 'Blues'

print(f"✓ Colormap: {COLORMAP_NAME}")
print(f"  (change COLORMAP_NAME to any matplotlib colormap name)")

## Database Access Functions

In [ ]:
@contextmanager
def get_db_cursor():
    """Context manager for database cursor."""
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

def get_available_variables():
    """Get all unique variables in the database."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT DISTINCT variable FROM climate_grid ORDER BY variable"
        )
        return [row['variable'] for row in cur.fetchall()]

def get_available_times(variable):
    """Get all unique times for a variable."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT DISTINCT time FROM climate_grid WHERE variable = %s ORDER BY time",
            [variable]
        )
        return [row['time'] for row in cur.fetchall()]

def get_grid_data(variable, time):
    """Load grid data for a variable and time."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT lat, lon, value FROM climate_grid WHERE variable = %s AND time::text LIKE %s ORDER BY lat, lon",
            [variable, f"{time}%"]
        )
        rows = cur.fetchall()
        
        if not rows:
            return None
        
        lats = np.array([row['lat'] for row in rows])
        lons = np.array([row['lon'] for row in rows])
        values = np.array([row['value'] for row in rows])
        
        return {
            'lats': lats,
            'lons': lons,
            'values': values,
        }

def get_global_min_max(variable):
    """Get global min/max across all times for a variable."""
    with get_db_cursor() as cur:
        cur.execute(
            "SELECT MIN(value) as min_val, MAX(value) as max_val FROM climate_grid WHERE variable = %s",
            [variable]
        )
        row = cur.fetchone()
        return float(row['min_val']), float(row['max_val'])

print("✓ Database functions defined")

## COG Generation Functions

In [ ]:
# apply_colormap(values_normalized, COLORMAP_NAME) is imported from
# app.utils.colormap_utils — no local colormap function needed.

def normalize_values(values, data_min, data_max):
    """
    Normalize values to range [0, 1] using global min/max.
    Also returns an alpha channel (255 for valid data, 0 for NaN/transparent).
    """
    # Track NaN locations for alpha channel
    nan_mask = np.isnan(values)
    alpha = np.where(nan_mask, 0, 255).astype(np.uint8)
    
    if data_max == data_min:
        normalized = np.full_like(values, 0.5, dtype=float)
        grayscale = np.full_like(values, 127, dtype=np.uint8)
    else:
        normalized = (values - data_min) / (data_max - data_min)
        normalized = np.clip(normalized, 0, 1)
        grayscale = (normalized * 255).astype(np.uint8)
    
    # Zero out NaN values in grayscale to avoid artifacts
    grayscale[nan_mask] = 0
    return normalized, grayscale, alpha

def rasterize_grid_data_native(lats, lons, values, bounds):
    """
    Rasterize grid data at its NATIVE resolution using vectorized numpy.
    
    IMPORTANT: This function properly handles NON-UNIFORM grids by:
    1. Using actual lat/lon values to determine pixel positions (not assuming uniform spacing)
    2. Mapping each data point to its correct position within the bounds
    3. Filling gaps with NaN for missing data areas
    
    Returns:
        (H, W) rasterized grid at native resolution
    """
    # Get unique lat/lon values sorted correctly
    unique_lats = np.sort(np.unique(lats))[::-1]  # descending (north to south)
    unique_lons = np.sort(np.unique(lons))         # ascending (west to east)
    
    native_h = len(unique_lats)
    native_w = len(unique_lons)
    
    print(f"  - Native grid resolution: {native_w}x{native_h}")
    
    # Create lookup maps for lat/lon -> index
    lat_to_idx = {lat: idx for idx, lat in enumerate(unique_lats)}
    lon_to_idx = {lon: idx for idx, lon in enumerate(unique_lons)}
    
    # Vectorized index mapping
    y_indices = np.array([lat_to_idx.get(lat, -1) for lat in lats])
    x_indices = np.array([lon_to_idx.get(lon, -1) for lon in lons])
    
    # Filter valid points (must have valid indices and non-NaN values)
    valid = (y_indices >= 0) & (x_indices >= 0) & ~np.isnan(values)
    
    # Create output grid and fill using vectorized indexing
    pixels = np.full((native_h, native_w), np.nan, dtype=np.float32)
    pixels[y_indices[valid], x_indices[valid]] = values[valid]
    
    # IMPORTANT: Do NOT attempt to fill gaps with interpolation!
    # The function keeps NaN values for missing data points.
    # These will be handled by the alpha channel (transparent) during rendering.
    
    return pixels

def resize_raster(native_raster, width, height):
    """
    Resize a native-resolution raster to target dimensions using PIL.
    NaN values are handled by filling with 0 before resize, then re-masking.
    """
    # Track NaN mask
    nan_mask = np.isnan(native_raster)
    
    # Fill NaNs for interpolation
    filled = native_raster.copy()
    filled[nan_mask] = 0
    
    # Resize data
    img = Image.fromarray(filled)
    resized = np.array(img.resize((width, height), Image.BILINEAR))
    
    # Resize NaN mask (nearest neighbor to preserve sharp edges)
    mask_img = Image.fromarray(nan_mask.astype(np.uint8) * 255)
    resized_mask = np.array(mask_img.resize((width, height), Image.NEAREST)) > 127
    
    # Re-apply NaN mask
    resized[resized_mask] = np.nan
    
    return resized

print("✓ Colormap, rasterization, and resize functions defined")

In [ ]:
def create_and_save_cog(variable, time, zoom_level, native_raster_wgs84, data_min, data_max, bounds_wgs84):
    """
    Create a Cloud-Optimized GeoTIFF in Web Mercator (EPSG:3857) with 5 bands:
    RGB visual + grayscale raw + alpha (transparency).

    FIX: native_raster_wgs84 is in WGS84/equirectangular space (pixels uniformly
    spaced in degrees). Naively tagging it as EPSG:3857 causes up to ~144 km of
    misalignment because Web Mercator Y is non-linear in latitude.

    This function FIRST reprojects the raw float32 data from WGS84 to EPSG:3857
    using rasterio.warp (which applies the correct Mercator latitude stretching),
    THEN applies normalisation and the configured colormap. This produces pixel content
    that matches the basemap tiles pixel-for-pixel.
    """
    native_h, native_w = native_raster_wgs84.shape
    target_w, target_h = ZOOM_SPECS[zoom_level]

    src_crs = CRS.from_epsg(4326)   # WGS84 - what the rasterised grid actually is
    dst_crs = CRS.from_epsg(3857)   # Web Mercator - what we want

    # ── Half-pixel correction ──────────────────────────────────────────
    # bounds_wgs84 values are cell CENTRES of the edge grid points.
    # from_bounds() treats them as pixel EDGES, so we must extend by
    # half a cell in every direction to avoid a systematic ~0.025° shift.
    cell_x = (bounds_wgs84['east'] - bounds_wgs84['west']) / (native_w - 1)
    cell_y = (bounds_wgs84['north'] - bounds_wgs84['south']) / (native_h - 1)
    adj = {
        'west':  bounds_wgs84['west']  - cell_x / 2,
        'east':  bounds_wgs84['east']  + cell_x / 2,
        'south': bounds_wgs84['south'] - cell_y / 2,
        'north': bounds_wgs84['north'] + cell_y / 2,
    }

    # Affine transform that describes the WGS84 raster (degrees, uniform spacing)
    src_transform = from_bounds(
        adj['west'], adj['south'],
        adj['east'], adj['north'],
        native_w, native_h
    )

    # Compute the correct Web Mercator affine transform at target pixel dimensions
    dst_transform, _, _ = calculate_default_transform(
        src_crs, dst_crs,
        native_w, native_h,
        left=adj['west'],
        bottom=adj['south'],
        right=adj['east'],
        top=adj['north'],
        dst_width=target_w,
        dst_height=target_h,
    )

    # --- Reproject raw float data WGS84 -> Web Mercator ---
    NODATA = np.float32(-9999.0)
    src_float = native_raster_wgs84.astype(np.float32)
    src_float[np.isnan(src_float)] = NODATA

    reprojected = np.full((target_h, target_w), NODATA, dtype=np.float32)
    reproject(
        source=src_float,
        destination=reprojected,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        src_nodata=NODATA,
        dst_nodata=NODATA,
        resampling=Resampling.bilinear,
    )
    del src_float

    # Restore NaN in reprojected data
    reprojected[reprojected == NODATA] = np.nan

    # --- Colorise the correctly-reprojected data ---
    normalized, grayscale, alpha = normalize_values(reprojected, data_min, data_max)
    del reprojected

    rgb_band = apply_colormap(normalized, COLORMAP_NAME)
    del normalized

    # --- Write output GeoTIFF ---
    output_dir = COG_BASE_DIR / variable / str(time)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"z{zoom_level}.tif"

    profile = {
        'driver': 'GTiff',
        'height': target_h,
        'width': target_w,
        'count': 5,  # RGB (3) + raw data (1) + alpha (1)
        'dtype': rasterio.uint8,
        'crs': dst_crs,
        'transform': dst_transform,
        'compress': 'lzw',
        'TILED': 'YES',
        'BLOCKXSIZE': 512,
        'BLOCKYSIZE': 512,
    }

    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(rgb_band[:, :, 0], 1)
        dst.write(rgb_band[:, :, 1], 2)
        dst.write(rgb_band[:, :, 2], 3)
        del rgb_band

        dst.write(grayscale, 4)
        del grayscale

        # Band 5: alpha (255 = valid data, 0 = transparent/nodata)
        dst.write(alpha, 5)
        del alpha

        dst.update_tags(
            VARIABLE=variable,
            TIME=str(time),
            ZOOM_LEVEL=str(zoom_level),
            DATA_MIN=str(data_min),
            DATA_MAX=str(data_max),
            COLORMAP_TYPE=COLORMAP_NAME,
            CRS='EPSG:3857',
            BOUNDS_WGS84=f"[{adj['west']}, {adj['south']}, {adj['east']}, {adj['north']}]",
        )

    return output_path

print("✓ COG creation function defined (proper WGS84→Web Mercator reprojection via rasterio.warp)")

## User Input

In [ ]:
# Get available variables
print("Fetching available variables from database...")
available_variables = get_available_variables()
print(f"\nAvailable variables: {available_variables}")

# User selects variable
selected_variable = available_variables[0]  # Default to first variable
print(f"\n✓ Selected variable: {selected_variable}")

In [ ]:
# Get available times for selected variable
print(f"Fetching available times for '{selected_variable}'...")
available_times = get_available_times(selected_variable)
print(f"Available times: {len(available_times)} entries")
print(f"First 5: {available_times[:5]}")
print(f"Last 5: {available_times[-5:]}")

# User selects time
selected_time = str(available_times[0])[:10]  # Use first time, extract date portion
print(f"\n✓ Selected time: {selected_time}")

## Load & Validate Data

In [ ]:
print(f"Loading grid data for {selected_variable} at {selected_time}...")
grid_data = get_grid_data(selected_variable, selected_time)

if grid_data is None:
    print(f"ERROR: No data found for {selected_variable} at {selected_time}")
else:
    lats = grid_data['lats']
    lons = grid_data['lons']
    values = grid_data['values']
    
    print(f"✓ Loaded {len(values)} grid points")
    print(f"  Latitude range: {lats.min():.2f} to {lats.max():.2f}")
    print(f"  Longitude range: {lons.min():.2f} to {lons.max():.2f}")
    print(f"  Value range: {values.min():.2f} to {values.max():.2f}")
    print(f"  NaN count: {np.isnan(values).sum()}")

## Compute Global Min/Max

In [ ]:
print(f"Computing global min/max across all times for '{selected_variable}'...")
global_min, global_max = get_global_min_max(selected_variable)

print(f"✓ Global range: {global_min:.2f} to {global_max:.2f}")
print(f"  This will be used for normalizing all COGs for this variable")

## Generate Progressive COGs (z0-z5)

In [ ]:
print(f"\nGenerating COGs for {selected_variable} - ALL TIME SLICES")
print(f"Global range: {global_min:.2f} to {global_max:.2f}")
print(f"Times to process: {len(available_times)}")
print("\n" + "="*60)

generated_files = []

for time_idx, time_val in enumerate(available_times):
    time_str = str(time_val)[:10]  # Extract date portion
    print(f"\n[{time_idx + 1}/{len(available_times)}] Processing {time_str}...")
    
    try:
        # Load grid data for this time
        grid_data = get_grid_data(selected_variable, time_str)
        if grid_data is None:
            print(f"  ✗ No data found for {time_str}, skipping")
            continue
        
        lats = grid_data['lats']
        lons = grid_data['lons']
        values = grid_data['values']
        
        # Step 1: Rasterize at native resolution
        print(f"  - Rasterizing at native resolution...")
        native_raster = rasterize_grid_data_native(
            lats, lons, values, AUSTRALIA_BOUNDS_WGS84
        )
        
        # Step 2: Generate each zoom level
        for zoom_level in sorted(ZOOM_SPECS.keys()):
            width, height = ZOOM_SPECS[zoom_level]
            
            try:
                # Create and save COG
                # Reprojection from WGS84 to Web Mercator is done inside create_and_save_cog
                output_path = create_and_save_cog(
                    selected_variable,
                    time_str,
                    zoom_level,
                    native_raster,
                    global_min,
                    global_max,
                    AUSTRALIA_BOUNDS_WGS84,
                )
                
                generated_files.append(output_path)
                
            except Exception as e:
                print(f"    ✗ ERROR at z{zoom_level}: {e}")
        
        print(f"  ✓ Generated 6 COGs (z0-z5) for {time_str}")
        del native_raster  # Free ~2-5 MB per time
        
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*60)
print(f"\n✓ Generated {len(generated_files)} COG files total")
print(f"✓ Coverage: {len(generated_files) // 6} time slices × 6 zoom levels")

## Verification & Summary

In [ ]:
print("\nGenerated Files Summary:")
print("="*60)

output_dir = COG_BASE_DIR / selected_variable / str(selected_time)
if output_dir.exists():
    files = sorted(output_dir.glob('z*.tif'))
    total_size_mb = 0
    
    for f in files:
        size_mb = f.stat().st_size / (1024**2)
        total_size_mb += size_mb
        print(f"{f.name:12} {size_mb:8.2f} MB")
    
    print("="*60)
    print(f"{'Total':12} {total_size_mb:8.2f} MB")
    print(f"Location: {output_dir}")
else:
    print("ERROR: Output directory not found!")

In [ ]:
# Test file properties
print("\nVerifying first COG file (z0)...")

test_file = COG_BASE_DIR / selected_variable / str(selected_time) / "z0.tif"

if test_file.exists():
    with rasterio.open(test_file) as src:
        print(f"✓ CRS: {src.crs}")
        print(f"✓ Size: {src.width}×{src.height}")
        print(f"✓ Bands: {src.count}")
        print(f"✓ Data type: {src.dtypes[0]}")
        print(f"✓ Bounds: {src.bounds}")
        print(f"✓ Metadata tags:")
        tags = src.tags()
        for key, value in tags.items():
            print(f"    {key}: {value}")
else:
    print(f"ERROR: Test file not found: {test_file}")

## Next Steps

The COGs have been generated successfully! 

**To use them:**
1. Start the FastAPI server: `python -m uvicorn app.main:app --reload`
2. Test the endpoints:
   - GET `/climate-mvt/variables` - List available variables
   - GET `/climate-mvt/times/{variable}` - List available times
   - GET `/climate-mvt/{variable}/{time}/z{zoom}.tif` - Download COG
3. In your DeckGL frontend, use BitmapLayer to consume the COGs

**To generate more COGs:**
- Change `selected_variable` and/or `selected_time` above and re-run the generation cells
- The notebook will generate all z0-z5 levels for each variable/time combination